# SQL one-shot experiment trail: exp002 -> exp003 -> exp004

This is an exploration notebook, not a postmortem. Run it top to bottom once, then change filters and case IDs.

Breadcrumbs to keep in mind:
- exp002 is useful history, but its eval files are 5-row samples.
- exp003 and exp004 use the same 25-row eval files, so that is the cleaner regression comparison.
- Lower train loss is not the same thing as better one-shot SQL execution accuracy.


In [ ]:
from collections import Counter
from pathlib import Path
import json
import math
import statistics

ROOT = Path.cwd()
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repo root."

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

def read_json(path):
    path = ROOT / path
    if not path.exists():
        return None
    return json.loads(path.read_text())

def read_jsonl(path):
    path = ROOT / path
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

def pct(x):
    if x is None:
        return ""
    return f"{100 * x:.1f}%"

def quantile(values, q):
    values = sorted(v for v in values if v is not None)
    if not values:
        return None
    idx = min(len(values) - 1, max(0, math.ceil(q * len(values)) - 1))
    return values[idx]

def clip(text, n=110):
    if text is None:
        return ""
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[: n - 3] + "..."

def md_table(rows, columns):
    def render(value):
        if value is None:
            return ""
        if isinstance(value, float):
            return f"{value:.3f}"
        return str(value).replace("|", "\\|").replace("\n", "<br>")
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(render(row.get(col)) for col in columns) + " |")
    text = "\n".join(lines)
    if Markdown and display:
        display(Markdown(text))
    else:
        print(text)


## Experiment registry

First breadcrumb: before looking at pass rates, make sure you know which artifacts each run used. The names are doing useful work here.

In [ ]:
EXPS = {
    "exp002": {
        "id": "qwen35_0_8b__exp002_spider_bird_sft",
        "manifest": "experiments/sql/qwen35_0_8b__exp002_spider_bird_sft.json",
        "train_summary": "artifacts/sql/qwen35_0_8b__exp002_spider_bird_sft/train_summary.json",
        "train": [
            "datasets/sql/train/spider_train_sample_v1.jsonl",
            "datasets/sql/train/bird_train_sample_v1.jsonl",
        ],
        "eval_note": "sample eval; use as history, not apples-to-apples with exp003/004",
        "results": {
            ("spider", "base"): "results/sql/qwen35_0_8b__exp002_spider_bird_sft/base__spider_validation_sample_v1.json",
            ("spider", "adapter"): "results/sql/qwen35_0_8b__exp002_spider_bird_sft/adapter__spider_validation_sample_v1.json",
            ("bird", "base"): "results/sql/qwen35_0_8b__exp002_spider_bird_sft/base__bird_validation_sample_v1.json",
            ("bird", "adapter"): "results/sql/qwen35_0_8b__exp002_spider_bird_sft/adapter__bird_validation_sample_v1.json",
        },
    },
    "exp003": {
        "id": "qwen35_0_8b__exp003_one_shot_spider_bird_sft",
        "manifest": "experiments/sql/qwen35_0_8b__exp003_one_shot_spider_bird_sft.json",
        "train_summary": "artifacts/sql/qwen35_0_8b__exp003_one_shot_spider_bird_sft/train_summary.json",
        "train": [
            "datasets/sql/train/spider_train_100_v1.jsonl",
            "datasets/sql/train/bird_train_100_v1.jsonl",
        ],
        "eval_note": "fixed 25-row eval",
        "results": {
            ("spider", "base"): "results/sql/qwen35_0_8b__exp003_one_shot_spider_bird_sft/base__spider_validation_25_v1.json",
            ("spider", "adapter"): "results/sql/qwen35_0_8b__exp003_one_shot_spider_bird_sft/adapter__spider_validation_25_v1.json",
            ("bird", "base"): "results/sql/qwen35_0_8b__exp003_one_shot_spider_bird_sft/base__bird_validation_25_v1.json",
            ("bird", "adapter"): "results/sql/qwen35_0_8b__exp003_one_shot_spider_bird_sft/adapter__bird_validation_25_v1.json",
        },
    },
    "exp004": {
        "id": "qwen35_0_8b__exp004_one_shot_spider_bird_sft",
        "manifest": "experiments/sql/qwen35_0_8b__exp004_one_shot_spider_bird_sft.json",
        "train_summary": "artifacts/sql/qwen35_0_8b__exp004_one_shot_spider_bird_sft/train_summary.json",
        "train": [
            "datasets/sql/train/spider_train_250_v1.jsonl",
            "datasets/sql/train/bird_train_250_v1.jsonl",
        ],
        "eval_note": "same fixed 25-row eval as exp003; adapter rerun",
        "results": {
            ("spider", "adapter"): "results/sql/qwen35_0_8b__exp004_one_shot_spider_bird_sft/adapter__spider_validation_25_v1.json",
            ("bird", "adapter"): "results/sql/qwen35_0_8b__exp004_one_shot_spider_bird_sft/adapter__bird_validation_25_v1.json",
        },
    },
}

rows = []
for exp, spec in EXPS.items():
    rows.append({
        "exp": exp,
        "manifest": (ROOT / spec["manifest"]).exists(),
        "train_summary": (ROOT / spec["train_summary"]).exists(),
        "result_files": sum((ROOT / p).exists() for p in spec["results"].values()),
        "eval_note": spec["eval_note"],
    })
md_table(rows, ["exp", "manifest", "train_summary", "result_files", "eval_note"])


## Manifest and training contract

Look for changes that affect the experiment contract, not only metric movement. If two things changed at once, do not assign causality too quickly.

In [ ]:
rows = []
for exp, spec in EXPS.items():
    manifest = read_json(spec["manifest"]) or {}
    summary = read_json(spec["train_summary"]) or {}
    trainer = manifest.get("trainer", {})
    method = manifest.get("training_method", {})
    metrics = summary.get("training_metrics", {})
    rows.append({
        "exp": exp,
        "stage": method.get("stage"),
        "loss_target": method.get("loss_target"),
        "train_rows": summary.get("train_row_count", sum(len(read_jsonl(p)) for p in spec["train"])),
        "lr": trainer.get("learning_rate"),
        "epochs": trainer.get("num_train_epochs"),
        "lora_r": (manifest.get("lora") or {}).get("r"),
        "train_loss": metrics.get("train_loss"),
        "runtime_sec": metrics.get("train_runtime"),
    })
md_table(rows, ["exp", "stage", "loss_target", "train_rows", "lr", "epochs", "lora_r", "train_loss", "runtime_sec"])


## Data shape

Breadcrumb: bigger is not automatically broader. Check row counts, DB coverage, schema length, SQL length, and whether knowledge text is present.

In [ ]:
def dataset_stats(path):
    data = read_jsonl(path)
    schema_lens = [len(row.get("schema_text") or "") for row in data]
    sql_lens = [len(row.get("target_sql") or row.get("gold_sql") or "") for row in data]
    question_lens = [len(row.get("question") or "") for row in data]
    dbs = Counter(row.get("db_id") for row in data)
    return {
        "rows": len(data),
        "benchmark": data[0].get("source_benchmark") if data else "",
        "db_count": len(dbs),
        "top_db": dbs.most_common(1)[0][0] if dbs else "",
        "knowledge_rows": sum(1 for row in data if row.get("knowledge_text")),
        "schema_med": int(statistics.median(schema_lens)) if schema_lens else None,
        "schema_p90": quantile(schema_lens, 0.90),
        "question_med": int(statistics.median(question_lens)) if question_lens else None,
        "sql_med": int(statistics.median(sql_lens)) if sql_lens else None,
    }

rows = []
for exp, spec in EXPS.items():
    for path in spec["train"]:
        stats = dataset_stats(path)
        rows.append({"exp": exp, "file": Path(path).name, **stats})
md_table(rows, ["exp", "file", "benchmark", "rows", "db_count", "top_db", "knowledge_rows", "schema_med", "schema_p90", "question_med", "sql_med"])


## Result summaries

Breadcrumb: separate base-vs-adapter improvement from run-vs-run movement. exp004 did not rerun base because the eval files are the same as exp003.

In [ ]:
def analysis_path_for(result_path):
    return result_path.replace(".json", ".analysis.json")

def top_failures(path):
    analysis = read_json(analysis_path_for(path)) or {}
    counts = analysis.get("failure_counts") or {}
    return ", ".join(f"{k}:{v}" for k, v in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])))

rows = []
for exp, spec in EXPS.items():
    for (dataset, variant), path in spec["results"].items():
        result = read_json(path)
        if not result:
            continue
        rows.append({
            "exp": exp,
            "dataset": dataset,
            "variant": variant,
            "passed": result.get("passed_count"),
            "cases": result.get("case_count"),
            "pass_rate": pct(result.get("pass_rate")),
            "eval_file": Path(result.get("eval_dataset", "")).name,
            "failure_counts": top_failures(path),
        })
md_table(rows, ["exp", "dataset", "variant", "passed", "cases", "pass_rate", "eval_file", "failure_counts"])


## Deltas to inspect

Tiny note: exp002 -> exp003 has a tempting improvement story, but the eval set changed at the same time. Treat it as directional history.

For exp003 -> exp004, the eval files stayed fixed. That makes the adapter movement worth inspecting case by case.

In [ ]:
def result(exp, dataset, variant="adapter"):
    path = EXPS[exp]["results"].get((dataset, variant))
    return read_json(path) if path else None

comparisons = [
    ("history_only", "exp002", "exp003", False),
    ("fixed_eval", "exp003", "exp004", True),
]

rows = []
for label, left, right, comparable in comparisons:
    for dataset in ["spider", "bird"]:
        a = result(left, dataset, "adapter")
        b = result(right, dataset, "adapter")
        if not a or not b:
            continue
        rows.append({
            "comparison": label,
            "dataset": dataset,
            "from": left,
            "to": right,
            "from_score": f"{a['passed_count']}/{a['case_count']}",
            "to_score": f"{b['passed_count']}/{b['case_count']}",
            "delta_passes": b["passed_count"] - a["passed_count"],
            "delta_rate_pp": round(100 * (b["pass_rate"] - a["pass_rate"]), 1),
            "same_eval": comparable,
        })
md_table(rows, ["comparison", "dataset", "from", "to", "from_score", "to_score", "delta_passes", "delta_rate_pp", "same_eval"])


## Failure bucket movement

Breadcrumb: when BIRD moves the wrong way, ask whether it became more syntactically broken, more schema-wrong, or mostly result-wrong.

In [ ]:
def failure_counts(exp, dataset, variant="adapter"):
    path = EXPS[exp]["results"].get((dataset, variant))
    if not path:
        return Counter()
    analysis = read_json(analysis_path_for(path)) or {}
    return Counter(analysis.get("failure_counts") or {})

rows = []
for dataset in ["spider", "bird"]:
    c3 = failure_counts("exp003", dataset)
    c4 = failure_counts("exp004", dataset)
    for bucket in sorted(set(c3) | set(c4)):
        rows.append({
            "dataset": dataset,
            "bucket": bucket,
            "exp003": c3.get(bucket, 0),
            "exp004": c4.get(bucket, 0),
            "delta": c4.get(bucket, 0) - c3.get(bucket, 0),
        })
md_table(rows, ["dataset", "bucket", "exp003", "exp004", "delta"])


## Case flips

This is where the useful debugging usually starts. Do not read all failures first; start with cases that changed behavior between exp003 and exp004.

In [ ]:
EVAL_FILES = {
    "spider": "datasets/sql/eval/spider_validation_25_v1.jsonl",
    "bird": "datasets/sql/eval/bird_validation_25_v1.jsonl",
}

def records_by_case(exp, dataset):
    res = result(exp, dataset, "adapter") or {}
    return {row["case_id"]: row for row in res.get("records", [])}

def eval_by_case(dataset):
    return {row["case_id"]: row for row in read_jsonl(EVAL_FILES[dataset])}

def flip_rows(dataset):
    r3 = records_by_case("exp003", dataset)
    r4 = records_by_case("exp004", dataset)
    eval_cases = eval_by_case(dataset)
    rows = []
    for case_id in sorted(set(r3) & set(r4)):
        if bool(r3[case_id].get("passed")) == bool(r4[case_id].get("passed")):
            continue
        case = eval_cases.get(case_id, {})
        rows.append({
            "case_id": case_id,
            "direction": "pass->fail" if r3[case_id].get("passed") else "fail->pass",
            "question": clip(case.get("question"), 95),
            "gold": clip(case.get("gold_sql"), 95),
            "exp003_sql": clip(r3[case_id].get("predicted_sql"), 95),
            "exp004_sql": clip(r4[case_id].get("predicted_sql"), 95),
            "exp004_error": clip(r4[case_id].get("prediction_error"), 80),
        })
    return rows

for dataset in ["spider", "bird"]:
    print(f"\n{dataset.upper()} flips")
    md_table(flip_rows(dataset), ["case_id", "direction", "question", "gold", "exp003_sql", "exp004_sql", "exp004_error"])


## Single-case microscope

Pick a flip above, then inspect the full prompt ingredients. Bread crumb: for BIRD, pay attention to quoted column names, knowledge text, and whether the model invented a nearby schema name.

In [ ]:
DATASET = "bird"
CASE_ID = "bird_validation_00001"  # Change this after looking at the flip table.

case = eval_by_case(DATASET).get(CASE_ID, {})
r3 = records_by_case("exp003", DATASET).get(CASE_ID, {})
r4 = records_by_case("exp004", DATASET).get(CASE_ID, {})

print("QUESTION:\n", case.get("question"))
print("\nKNOWLEDGE:\n", case.get("knowledge_text"))
print("\nGOLD SQL:\n", case.get("gold_sql"))
print("\nEXP003:", "PASS" if r3.get("passed") else "FAIL", "|", r3.get("prediction_error"))
print(r3.get("predicted_sql"))
print("\nEXP004:", "PASS" if r4.get("passed") else "FAIL", "|", r4.get("prediction_error"))
print(r4.get("predicted_sql"))
print("\nSCHEMA PREVIEW:\n", clip(case.get("schema_text"), 1200))


## Notes to leave for the next run

Use this cell as a scratchpad before designing exp005.

- What changed from exp002 to exp003 that is definitely real?
- What changed from exp002 to exp003 that is confounded by the eval change?
- Which exp003 -> exp004 BIRD flips are syntax/quoting failures?
- Which failures look like schema grounding rather than model capacity?
- If you trained only one narrow slice next, what would the rows need to teach?
